<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/05_week/calibration_ece.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
import torch.nn.functional as F


def compute_ece(logits, labels, n_bins=10):
    """TR: Logit değerlerinden Expected Calibration Error (ECE) skorunu hesaplar.

    EN: Calculates Expected Calibration Error (ECE) from raw logits and true
    labels.
    """
    # TR: Logitleri Softmax ile 0-1 arasındaki olasılıklara dönüştürüyoruz.
    # EN: Convert raw logits to probabilities using Softmax.
    probs = F.softmax(logits, dim=1).detach().numpy()
    labels = labels.numpy() if isinstance(labels, torch.Tensor) else labels

    # TR: En yüksek olasılık değerini özgüven (confidence) olarak alıyoruz.
    # EN: Extract maximum probability as confidence score for each prediction.
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == labels

    # TR: 0 ile 1 aralığını n_bins adet kovana bölüyoruz.
    # EN: Divide [0, 1] confidence range into n_bins equal intervals.
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i + 1]

        # TR: İlgili özgüven aralığına düşen tahminleri filtreliyoruz.
        # EN: Filter predictions falling within the current confidence bin.
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            # TR: Kovandaki doğruluk ve özgüven ortalamalarını hesaplıyoruz.
            # EN: Calculate average accuracy and confidence inside the bin.
            accuracy_in_bin = np.mean(accuracies[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])

            # TR: Ağırlıklı mutlak farkı ECE skoruna ekliyoruz.
            # EN: Accumulate weighted absolute error to final ECE score.
            ece += np.abs(accuracy_in_bin - avg_confidence_in_bin) * prop_in_bin

    return ece